# Train v1 
**Basic config:**
- Config for lightgbm training is conservative.
- Split 80% data for training.
- Delete sub_code from training cause it has high gain importance however the overlap between train and test is onlu 23% which is dangerous.

In [ ]:
from pathlib import Path 
import sys


In [ ]:
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))


In [ ]:
import lightgbm as lgb
import pandas as pd
import gc
from src.score import weighted_rmse_score as score 


In [ ]:
def set_cat(df,cat_features):
    for feat in cat_features:
        df[feat]=df[feat].astype('category')
    return df


In [ ]:
df=pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
cat_features = ['code','sub_category','horizon']
features=[feat for feat in df.columns if feat not in ['id','sub_code','ts_index','weight','y_target']]
df=set_cat(df,cat_features)


In [10]:
split_percent = 0.8
split_index=int(df['ts_index'].max()*split_percent)
train_df = df[df['ts_index']<=split_index]
val_df = df[df['ts_index']>split_index]


In [ ]:
del df
gc.collect()


In [11]:
dtrain=lgb.Dataset(train_df[features],
                   label=train_df['y_target'],
                   weight=train_df['weight'],
                   categorical_feature=cat_features,
                   free_raw_data=False)

dval=lgb.Dataset(val_df[features],
                 label=val_df['y_target'],
                 weight=val_df['weight'],
                categorical_feature=cat_features,
                reference=dtrain,
                free_raw_data=False)


In [ ]:
params = {
    'objective':'regression',
    'boosting_type':'gbdt',
    'metric':'rmse',# root mean squared error
    'num_leaves':32,
    'learning_rate':0.01,
    'bagging_fraction':0.7,#pick 70% of data to train each tree
    'bagging_freq':5,#perform bagging every 5 iterations
    'feature_fraction':0.8,#pick 70% of features to train each tree
    'seed':42,
    'verbosity':-1
}


In [7]:
model=lgb.train(params,
                dtrain,
                valid_sets=[dtrain,dval],
                valid_names=['train','val'],
                num_boost_round=1000,
                callbacks=[lgb.early_stopping(stopping_rounds=50),
                           lgb.log_evaluation(period=50)]
                )


Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 0.00203718	val's rmse: 0.00263204
[100]	train's rmse: 0.00202276	val's rmse: 0.00262905
[150]	train's rmse: 0.00201141	val's rmse: 0.00262696
[200]	train's rmse: 0.00200115	val's rmse: 0.0026253
[250]	train's rmse: 0.00199148	val's rmse: 0.00265603
Early stopping, best iteration is:
[211]	train's rmse: 0.00199919	val's rmse: 0.00262484


In [8]:
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print(importance.head(10))


         feature    importance
2        horizon  7.636466e+07
14     feature_l  5.258850e+07
1   sub_category  4.311518e+07
24     feature_v  3.885813e+07
87    feature_cg  3.562070e+07
61    feature_bg  3.292366e+07
15     feature_m  3.229169e+07
70    feature_bp  3.204266e+07
45    feature_aq  3.141135e+07
47    feature_as  2.852958e+07


In [ ]:
zero_gain_features=importance[importance['importance']==0]['feature'].tolist()
print(zero_gain_features)


['feature_d', 'feature_e', 'feature_s', 'feature_o', 'feature_g', 'feature_ax', 'feature_av', 'feature_aa', 'feature_w', 'feature_ab', 'feature_bb', 'feature_ba', 'feature_aw', 'feature_bj', 'feature_bi', 'feature_bc', 'feature_bu', 'feature_bv']


In [ ]:
# Saving the model
model.save_model(PROJECT_ROOT / "models" / "lgb_model_v1.txt", num_iteration=model.best_iteration)


: 

In [ ]:
# Loading the model later
trained_model = lgb.Booster(model_file= str(PROJECT_ROOT / "models" / "lgb_model_v1.txt"))

train_preds = trained_model.predict(train_df[features], num_iteration=trained_model.best_iteration)
val_preds = trained_model.predict(val_df[features], num_iteration=trained_model.best_iteration)

train_score = score(train_df['y_target'], train_preds, train_df['weight'])
val_score = score(val_df['y_target'], val_preds, val_df['weight'])


In [ ]:
print(f"Train Score: {train_score:.4f}")
print(f"Validation Score: {val_score:.4f}")


Train Score: 0.2460
Validation Score: 0.1080


: 